# Wall-pump width endpoints S100 v1

Generate the missing width-controlled endpoint frames and the small CPU/GPU bridge ensemble. Results are five-trajectory shards; execution batches are chosen by an A100 throughput/memory benchmark and checkpoint every five cycles.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Locked scientific configuration and run controls


In [ ]:
from pathlib import Path

BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/11_wall_pump_width_endpoints')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/wall_pump_width_endpoints_s100_v1')
LOCAL_BUNDLE_DIR = Path('/content/11_wall_pump_width_endpoints')
SCRATCH_ROOT = Path('/content/wall_pump_width_endpoints_s100_v1_scratch')
REPORT_ONLY = False
BENCHMARK_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None

CONFIG = {'Nx_values': [20, 24, 28, 32],
 'Ny': 24,
 'benchmark': {'execution_batch_candidates': [10, 20, 40, 60, 80, 100],
               'full_timing_samples': 5,
               'local_56_core_forecast_hours': 80.0,
               'maximum_a100_forecast_hours': 40.0,
               'minimum_gpu_headroom_gib': 8.0,
               'short_cycles': 5,
               'worst_case': {'Nx': 32, 'protocol': 'dense', 'wall': 'soft'}},
 'bridge_cells': [{'Nx': 20, 'protocol': 'nsh1'},
                  {'Nx': 24, 'protocol': 'nsh1'},
                  {'Nx': 20, 'protocol': 'dense'}],
 'checkpoint_cycles': 5,
 'cycles': 48,
 'device': 'cuda:0',
 'dtype': 'complex128',
 'execution_backend': 'gpu',
 'primary_cells': [{'Nx': 28, 'protocol': 'nsh1'},
                   {'Nx': 32, 'protocol': 'nsh1'},
                   {'Nx': 24, 'protocol': 'dense'},
                   {'Nx': 28, 'protocol': 'dense'},
                   {'Nx': 32, 'protocol': 'dense'}],
 'protocol': {'DW': True,
              'alpha_1': 1.0,
              'alpha_2': 30.0,
              'filling_frac': 0.5,
              'frame_reorthonormalize_interval': 1,
              'init_mode': 'default',
              'n_a': 0.5,
              'perfect_correction': True,
              'postselect': False,
              'postselect_probability': 0.0,
              'sequence': 'raster_y',
              'state_representation': 'physical_frame',
              'trial_orbitals': 'X',
              'triv_region_local_mode': False,
              'wall_locations_rule': ['Nx/4', '3Nx/4']},
 'result_shard_size': 5,
 'root_seed': 2026090701,
 'samples_per_bridge_wall': 25,
 'samples_per_primary_wall': 100,
 'sampling_revision': 'wall_pump_width_endpoints_s100_v1',
 'shells': {'dense': None, 'nsh1': 1},
 'walls': {'hard': {'dw_truncation': True, 'meas_slab_only': True},
           'soft': {'dw_truncation': False, 'meas_slab_only': False}}}

print(f'Revision: {CONFIG["sampling_revision"]}')
print('Workload: 1,000 primary + 150 bridge endpoints; 230 five-sample shards')
print('Scientific contract: Ny=24, 48 cycles, raster_y, pure half filling, perfect correction')
print(f'Bundle: {BUNDLE_DRIVE_DIR}')
print(f'Output: {OUTPUT_ROOT}')
print(f'Scratch: {SCRATCH_ROOT}')


## 2. A100 and complex128 check


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required')
name = torch.cuda.get_device_name(0)
gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in name.upper() or gib < 38:
    raise RuntimeError(f'A 40-GB-class A100 is required; found {name}, {gib:.1f} GiB')
probe = torch.zeros(1, dtype=torch.complex128, device='cuda:0')
del probe
print(f'Device: {name}; memory={gib:.1f} GiB; dtype=complex128')


## 3. Stage locally and run/resume

The child output is streamed so the outer shard bar and inner cycle bar remain visible. A restart verifies every result/completion pair and restores the exact last five-cycle state/RNG checkpoint for the current execution batch.


In [ ]:
import codecs, json, os, shutil, subprocess, sys

def run_streaming_child(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            raw = os.read(process.stdout.fileno(), 4096)
            if not raw:
                break
            sys.stdout.write(decoder.decode(raw)); sys.stdout.flush()
        tail = decoder.decode(b'', final=True)
        if tail:
            sys.stdout.write(tail); sys.stdout.flush()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
        raise
    finally:
        process.stdout.close()
    returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(BUNDLE_DRIVE_DIR)
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/wall_pump_width_endpoints_s100_v1_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR/'run_campaign.py'), '--config', str(config_path), '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY:
    command.append('--report-only')
if BENCHMARK_ONLY:
    command.append('--benchmark-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    command += ['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))]
print('[launch]', ' '.join(command), flush=True)
run_streaming_child(command)
print('[notebook] runner exited successfully', flush=True)


## 4. Release the runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
